<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/08_mlops_fastapi_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 08: MLOps — Deploy a Model with FastAPI

> **Goal:** Go from a trained scikit-learn model to a production-ready REST API with proper error handling, logging, and Docker deployment.

**What you'll build:**
1. Train and save a model
2. Build a FastAPI serving app
3. Add Pydantic validation, health checks, and logging
4. Write tests
5. Containerize with Docker

**Time:** ~4 hours

**Skills:** FastAPI, Pydantic, pytest, Docker, joblib

## Part 1: Train and Save the Model

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report

# Load Titanic data
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(URL)

# Feature engineering
df['Title'] = df['Name'].str.extract(r',\s*(\w+)\.', expand=False)
df['Title'] = df['Title'].map({
    'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'
}).fillna('Other')
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

FEATURES = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']
TARGET = 'Survived'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build pipeline
numeric_features = ['Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Embarked', 'Title']
ordinal_features = ['Pclass']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
    ('ord', SimpleImputer(strategy='most_frequent'), ordinal_features),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)),
])

pipeline.fit(X_train, y_train)

# Evaluate
auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
print(f'Test AUC: {auc:.4f}')
print(classification_report(y_test, pipeline.predict(X_test), target_names=['No Survive', 'Survive']))

# Save
os.makedirs('models', exist_ok=True)
MODEL_PATH = 'models/titanic_model_v1.pkl'
MODEL_META = {
    'version': '1.0.0',
    'features': FEATURES,
    'target': TARGET,
    'test_auc': round(auc, 4),
    'training_samples': len(X_train),
}
joblib.dump({'model': pipeline, 'meta': MODEL_META}, MODEL_PATH)
print(f'\nModel saved to {MODEL_PATH}')

## Part 2: FastAPI Application

Write these files to disk — we'll run them as a service.

In [ ]:
# Write the FastAPI app to main.py
app_code = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import joblib
import pandas as pd
import logging
import time

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

MODEL_PATH = "models/titanic_model_v1.pkl"
model_data = {}  # Will hold loaded model + metadata


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load model at startup, clean up at shutdown."""
    logger.info(f"Loading model from {MODEL_PATH}...")
    try:
        loaded = joblib.load(MODEL_PATH)
        model_data["model"] = loaded["model"]
        model_data["meta"] = loaded["meta"]
        logger.info(f"Model v{model_data['meta']['version']} loaded (AUC={model_data['meta']['test_auc']})")
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise
    yield
    logger.info("Shutting down...")


app = FastAPI(
    title="Titanic Survival Predictor API",
    description="Predicts survival probability for Titanic passengers",
    version="1.0.0",
    lifespan=lifespan,
)


# ── Request/Response schemas ───────────────────────────────────
class PassengerInput(BaseModel):
    pclass: int = Field(..., ge=1, le=3, description="Passenger class (1=First, 2=Second, 3=Third)")
    sex: str = Field(..., pattern="^(male|female)$")
    age: float = Field(..., ge=0, le=120)
    fare: float = Field(..., ge=0)
    embarked: Optional[str] = Field("S", pattern="^(S|C|Q)$")
    title: Optional[str] = Field("Mr", description="Mr, Mrs, Miss, Master, or Other")
    family_size: Optional[int] = Field(1, ge=1, le=20)
    is_alone: Optional[int] = Field(1, ge=0, le=1)

    class Config:
        json_schema_extra = {
            "example": {
                "pclass": 1,
                "sex": "female",
                "age": 29,
                "fare": 211.34,
                "embarked": "S",
                "title": "Miss",
                "family_size": 1,
                "is_alone": 1,
            }
        }


class PredictionResponse(BaseModel):
    survived: bool
    survival_probability: float
    confidence_level: str  # high / medium / low
    model_version: str


# ── Endpoints ─────────────────────────────────────────────────
@app.get("/health")
def health_check():
    if not model_data:
        raise HTTPException(status_code=503, detail="Model not loaded")
    return {
        "status": "healthy",
        "model_version": model_data["meta"]["version"],
        "model_auc": model_data["meta"]["test_auc"],
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(passenger: PassengerInput):
    if not model_data:
        raise HTTPException(status_code=503, detail="Model not available")

    start = time.perf_counter()

    # Build DataFrame matching training features
    data = pd.DataFrame([{
        "Pclass": passenger.pclass,
        "Sex": passenger.sex,
        "Age": passenger.age,
        "Fare": passenger.fare,
        "Embarked": passenger.embarked,
        "Title": passenger.title,
        "FamilySize": passenger.family_size,
        "IsAlone": passenger.is_alone,
    }])

    proba = float(model_data["model"].predict_proba(data)[0, 1])
    survived = proba >= 0.5

    # Confidence based on probability distance from threshold
    distance = abs(proba - 0.5)
    if distance > 0.35:
        confidence = "high"
    elif distance > 0.20:
        confidence = "medium"
    else:
        confidence = "low"

    elapsed_ms = (time.perf_counter() - start) * 1000
    logger.info(f"Prediction: survived={survived}, prob={proba:.3f}, latency={elapsed_ms:.1f}ms")

    return PredictionResponse(
        survived=survived,
        survival_probability=round(proba, 4),
        confidence_level=confidence,
        model_version=model_data["meta"]["version"],
    )


@app.get("/model-info")
def model_info():
    if not model_data:
        raise HTTPException(status_code=503, detail="Model not loaded")
    return model_data["meta"]
'''

with open('main.py', 'w') as f:
    f.write(app_code)

print('FastAPI app written to main.py')
print('Run with: uvicorn main:app --reload --port 8000')
print('Docs at:  http://localhost:8000/docs')

In [ ]:
# Write tests
test_code = '''
import pytest
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    data = response.json()
    assert data["status"] == "healthy"
    assert "model_version" in data


def test_predict_first_class_female():
    """1st class female should have high survival probability."""
    response = client.post("/predict", json={
        "pclass": 1, "sex": "female", "age": 25, "fare": 200.0,
        "title": "Miss", "family_size": 1, "is_alone": 1
    })
    assert response.status_code == 200
    data = response.json()
    assert data["survived"] == True
    assert data["survival_probability"] > 0.7


def test_predict_third_class_male():
    """3rd class adult male should have low survival probability."""
    response = client.post("/predict", json={
        "pclass": 3, "sex": "male", "age": 35, "fare": 7.5,
        "title": "Mr", "family_size": 1, "is_alone": 1
    })
    assert response.status_code == 200
    data = response.json()
    assert data["survived"] == False
    assert data["survival_probability"] < 0.4


def test_predict_invalid_pclass():
    """Pclass must be 1, 2, or 3."""
    response = client.post("/predict", json={
        "pclass": 5, "sex": "male", "age": 30, "fare": 50.0
    })
    assert response.status_code == 422  # Unprocessable Entity


def test_predict_invalid_sex():
    response = client.post("/predict", json={
        "pclass": 1, "sex": "unknown", "age": 30, "fare": 50.0
    })
    assert response.status_code == 422


def test_predict_missing_required_field():
    """Missing required fields should return 422."""
    response = client.post("/predict", json={"pclass": 1})
    assert response.status_code == 422


def test_model_info_endpoint():
    response = client.get("/model-info")
    assert response.status_code == 200
    data = response.json()
    assert "version" in data
    assert "test_auc" in data
    assert data["test_auc"] > 0.80  # Minimum quality gate


def test_prediction_response_schema():
    """Response must have all required fields."""
    response = client.post("/predict", json={
        "pclass": 2, "sex": "female", "age": 40, "fare": 30.0
    })
    assert response.status_code == 200
    data = response.json()
    assert "survived" in data
    assert "survival_probability" in data
    assert "confidence_level" in data
    assert data["confidence_level"] in ["high", "medium", "low"]
    assert 0.0 <= data["survival_probability"] <= 1.0
'''

with open('test_main.py', 'w') as f:
    f.write(test_code)

print('Tests written to test_main.py')
print('Run with: pytest test_main.py -v')

In [ ]:
# Write Dockerfile
dockerfile = '''FROM python:3.11-slim

# Security: non-root user
RUN useradd -m -u 1000 appuser

WORKDIR /app

# Install dependencies first (layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY --chown=appuser:appuser . .

USER appuser

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

requirements = '''fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic==2.8.0
scikit-learn==1.5.0
pandas==2.2.0
numpy==2.0.0
joblib==1.4.0
httpx==0.27.0
pytest==8.3.0
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('Dockerfile and requirements.txt written!')
print()
print('To build and run:')
print('  docker build -t titanic-api .')
print('  docker run -p 8000:8000 titanic-api')
print()
print('To test:')
print('  curl http://localhost:8000/health')
print('  curl -X POST http://localhost:8000/predict \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{"pclass": 1, "sex": "female", "age": 29, "fare": 211.34}\'')

In [ ]:
# Test the API locally (without Docker)
from fastapi.testclient import TestClient
import importlib
import sys

# Quick in-notebook test
if 'main' in sys.modules:
    del sys.modules['main']

import main
client = TestClient(main.app)

# Test health
health = client.get('/health')
print(f'Health: {health.json()}')

# Test predictions
test_cases = [
    {'pclass': 1, 'sex': 'female', 'age': 29, 'fare': 211.34, 'title': 'Miss'},
    {'pclass': 3, 'sex': 'male', 'age': 22, 'fare': 7.25, 'title': 'Mr'},
    {'pclass': 2, 'sex': 'female', 'age': 35, 'fare': 26.0, 'title': 'Mrs', 'family_size': 3, 'is_alone': 0},
]

print('\nPREDICTIONS')
print('=' * 60)
for tc in test_cases:
    resp = client.post('/predict', json=tc)
    data = resp.json()
    survived_str = '✅ SURVIVED' if data['survived'] else '❌ PERISHED'
    print(f"{tc['sex']:6} | Class {tc['pclass']} | Age {tc['age']:2} | {survived_str} ({data['survival_probability']:.1%}) [{data['confidence_level']}]")

## Summary: ML Serving Architecture

```
Training Pipeline          Serving Pipeline
──────────────────         ──────────────────
Data collection            HTTP Request
     ↓                          ↓
Feature engineering        Input Validation (Pydantic)
     ↓                          ↓
Model training             Feature Transform (same pipeline!)
     ↓                          ↓
Evaluation                 Model.predict_proba()
     ↓                          ↓
joblib.dump(pipeline)      Response Schema (Pydantic)
                                ↓
                           HTTP Response
```

**Key principle:** The same sklearn Pipeline handles both training-time transforms and serving-time transforms — no code duplication, no train-serve skew.

## Challenge
1. Add a `/predict/batch` endpoint that accepts a list of passengers and returns all predictions
2. Add Prometheus metrics: `prediction_count`, `prediction_latency_seconds`
3. Add an `/explain` endpoint that returns top 3 features influencing the prediction
4. Write a load test with `locust` to measure throughput (target: 100 req/s)